In [ ]:
# Install required packages
!pip install --upgrade torchao peft transformers accelerate kagglehub -q

In [ ]:
# Import libraries
import os
import json
import time
import shutil
from os import mkdir

import torch
import torch.nn as nn
import kagglehub

from torch.amp import GradScaler, autocast
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification
from peft import get_peft_model, LoraConfig
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from tqdm.notebook import tqdm

In [ ]:
# Device configuration with T4 optimizations
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

# Enable TF32 for better T4 performance (if CUDA available)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 optimizations enabled for T4 GPU")

# Optimized Configuration for T4 GPU
CONFIG = {
    "data_dir": "./data",
    "batch_size": 32,                    # Optimized for T4 (16GB VRAM)
    "gradient_accumulation_steps": 2,   # Effective batch size = 64
    "num_workers": 2,
    "num_classes": 2,
    "image_size": 224,
    "lr": 2e-4,                          # Higher LR for OneCycleLR
    "weight_decay": 0.01,                # Regularization
    "epochs": 5,                        # Increased with early stopping
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.1,
    "patience": 3,                       # Early stopping patience
    "max_grad_norm": 1.0,                # Gradient clipping
    "save_dir": "./saved_model_optimized",
    "model_name": "google/vit-base-patch16-224",
}

LABEL_MAP = {0: "FAKE", 1: "REAL"}

print(f"\n Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Using Device: cuda
TF32 optimizations enabled for T4 GPU

 Configuration:
  data_dir: ./data
  batch_size: 32
  gradient_accumulation_steps: 2
  num_workers: 2
  num_classes: 2
  image_size: 224
  lr: 0.0002
  weight_decay: 0.01
  epochs: 5
  lora_r: 32
  lora_alpha: 64
  lora_dropout: 0.1
  patience: 3
  max_grad_norm: 1.0
  save_dir: ./saved_model_optimized
  model_name: google/vit-base-patch16-224


# Method Definitions

In [ ]:
def download_data(config):
    """Downloads the CIFAKE dataset and moves it to the local data directory."""
    target_dir = config["data_dir"]

    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"Dataset already exists at {target_dir}")
        return target_dir

    print("Downloading dataset via kagglehub...")
    cache_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
    print("Dataset ready!")
    return cache_path

In [ ]:
def get_dataloaders(config):
    """Creates and returns training and testing dataloaders with enhanced augmentation."""

    # ENHANCED TRAINING AUGMENTATION
    # Critical for AI-generated image detection
    train_transform = transforms.Compose([
        transforms.Resize((config["image_size"], config["image_size"])),
        transforms.RandomResizedCrop(config["image_size"], scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),

        # Color augmentations help detect subtle AI artifacts
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomGrayscale(p=0.1),

        # Blur helps with compression artifact detection
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
        ], p=0.3),

        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),

        # Random erasing prevents overfitting to specific patterns
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
    ])

    # Standard test transform (no augmentation)
    test_transform = transforms.Compose([
        transforms.Resize((config["image_size"], config["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_dir = os.path.join(config["data_dir"], "train")
    test_dir = os.path.join(config["data_dir"], "test")

    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
    test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        pin_memory=True,
        prefetch_factor=2,
        persistent_workers=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=True,
        prefetch_factor=2,
        persistent_workers=True
    )

    print(f"\n Dataset Statistics:")
    print(f"  Training samples: {len(train_dataset):,}")
    print(f"  Testing samples: {len(test_dataset):,}")
    print(f"  Classes: {train_dataset.classes}")

    return train_loader, test_loader, train_dataset.class_to_idx

In [ ]:
def build_model(config, device):
    """
    Loads ViT-Base model and applies LoRA for efficient fine-tuning.

    Key fixes:
    1. Removed ignore_mismatched_sizes=True
    2. Uses correct LoRA target module names for ViT architecture
    3. Adds task_type specification for proper PEFT initialization
    """
    print("\n  Building model...")

    # Load model WITHOUT ignore_mismatched_sizes
    print(f"  Loading {config['model_name']}...")
    model = ViTForImageClassification.from_pretrained(
        config["model_name"],
        num_labels=config["num_classes"],
        ignore_mismatched_sizes=True,
        trust_remote_code=True,
    )

    # Configure LoRA with CORRECT module names
    print("  Configuring LoRA...")
    lora_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        modules_to_save=["classifier"],
        inference_mode=False,
    )

    # Apply LoRA with error handling
    try:
        model = get_peft_model(model, lora_config)
        print("  LoRA applied successfully")

    except ValueError as e:
        # If this fails, print the actual module names for debugging
        print(f"\n  LoRA application failed!")
        print(f"  Error: {str(e)}\n")
        print("  Available modules containing 'query', 'key', 'value':")

        # Print actual module names for user to debug
        for name, module in model.named_modules():
            if any(kw in name for kw in ['query', 'key', 'value']):
                print(f"    {name}")

        raise

    # Move to device
    model = model.to(device)

    # Print statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    frozen_params = total_params - trainable_params

    print(f"\n Model ready for training!")
    print(f"  Device:           {device}")
    print(f"  Model:            {config['model_name']}")
    print(f"  Num classes:      {config['num_classes']}")
    print(f"  LoRA rank (r):    {config['lora_r']}")
    print(f"  LoRA alpha:       {config['lora_alpha']}")
    print(f"  ├─ Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.3f}%)")
    print(f"  ├─ Frozen params:    {frozen_params:,} ({100 * frozen_params / total_params:.3f}%)")
    print(f"  └─ Total params:     {total_params:,}")

    return model

In [ ]:
def evaluate_epoch(model, loader, criterion, device, epoch, epochs):
    """Validates the model at the end of an epoch."""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    val_bar = tqdm(loader, desc=f" Validating Epoch {epoch}/{epochs}", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            with autocast(device_type=device.type):
                outputs = model(pixel_values=images).logits
                loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted")
    recall = recall_score(all_labels, all_preds, average="weighted")
    f1 = f1_score(all_labels, all_preds, average="weighted")

    return avg_loss, acc, precision, recall, f1

In [ ]:
def train_model(model, train_loader, test_loader, config, device, class_to_idx, label_map):
    """Enhanced training loop with all optimizations."""

    # Loss function
    criterion = nn.CrossEntropyLoss()

    # Optimizer with weight decay
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    # LEARNING RATE SCHEDULER with warmup
    scheduler = OneCycleLR(
        optimizer,
        max_lr=config["lr"],
        epochs=config["epochs"],
        steps_per_epoch=len(train_loader),
        pct_start=0.1,  # 10% warmup
        anneal_strategy='cos'
    )

    # Mixed precision scaler
    scaler = GradScaler(device.type)

    # Training tracking
    epochs = config["epochs"]
    best_val_f1 = 0.0
    patience_counter = 0
    accumulation_steps = config["gradient_accumulation_steps"]

    print(f"\n Starting Training")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: {config['batch_size']}")
    print(f"  Gradient accumulation: {accumulation_steps}")
    print(f"  Effective batch size: {config['batch_size'] * accumulation_steps}")
    print(f"  Early stopping patience: {config['patience']}")
    print("="*60)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        start_time = time.time()

        batch_bar = tqdm(train_loader, desc=f" Epoch {epoch}/{epochs}", leave=False)

        for idx, (images, labels) in enumerate(batch_bar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Forward pass with mixed precision
            with autocast(device_type=device.type):
                outputs = model(pixel_values=images).logits
                loss = criterion(outputs, labels) / accumulation_steps  # Scale loss

            # Backward pass
            scaler.scale(loss).backward()

            # GRADIENT ACCUMULATION
            if (idx + 1) % accumulation_steps == 0:
                # GRADIENT CLIPPING for stability
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])

                # Optimizer step
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                # Learning rate scheduling
                scheduler.step()

            # Metrics
            total_loss += loss.item() * accumulation_steps
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Update progress bar
            current_lr = scheduler.get_last_lr()[0]
            batch_bar.set_postfix({
                'loss': f'{loss.item() * accumulation_steps:.4f}',
                'acc': f'{100 * correct / total:.2f}%',
                'lr': f'{current_lr:.2e}'
            })

        # Epoch metrics
        train_acc = correct / total
        train_loss = total_loss / len(train_loader)
        epoch_time = time.time() - start_time

        # Validation
        val_loss, val_acc, val_precision, val_recall, val_f1 = evaluate_epoch(
            model, test_loader, criterion, device, epoch, epochs
        )

        # Print epoch summary
        print(f"\n Epoch [{epoch}/{epochs}] — {epoch_time:.1f}s")
        print(f"  Train → Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"  Val   → Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
        print(f"  Val   → Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}")

        # EARLY STOPPING & BEST MODEL SAVING
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0

            # Save best model
            best_model_path = os.path.join(config["save_dir"], "best_model")
            os.makedirs(best_model_path, exist_ok=True)
            model.save_pretrained(best_model_path)

            # Save config with best model
            config_path = os.path.join(best_model_path, "training_config.json")
            save_config = {
                **config,
                "label_map": label_map,
                "class_to_idx": class_to_idx,
                "best_val_f1": float(best_val_f1),
                "best_epoch": epoch
            }
            with open(config_path, "w") as f:
                json.dump(save_config, f, indent=4)

            print(f"   New best F1: {val_f1:.4f} - Model saved!")
        else:
            patience_counter += 1
            print(f"   Patience: {patience_counter}/{config['patience']}")

        # Early stopping check
        if patience_counter >= config["patience"]:
            print(f"\n Early stopping triggered after {epoch} epochs")
            print(f"   Best F1 score: {best_val_f1:.4f}")
            break

    # Final model save (last checkpoint)
    final_model_path = os.path.join(config["save_dir"], "final_model")
    os.makedirs(final_model_path, exist_ok=True)
    model.save_pretrained(final_model_path)

    print(f"\n Training Complete!")
    print(f"  Best model saved to: {os.path.join(config['save_dir'], 'best_model')}")
    print(f"  Final model saved to: {final_model_path}")
    print(f"  Best validation F1: {best_val_f1:.4f}")

    return model

In [ ]:
def final_evaluation(model, test_loader, device, label_map):
    """Comprehensive final evaluation with detailed metrics."""
    print("\n" + "="*60)
    print(" FINAL EVALUATION")
    print("="*60)

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(device_type=device.type):
                outputs = model(pixel_values=images).logits

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted")
    recall = recall_score(all_labels, all_preds, average="weighted")
    f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"\n Overall Metrics:")
    print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")

    print(f"\n Detailed Classification Report:")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=[label_map[i] for i in sorted(label_map.keys())],
        digits=4
    ))

    return accuracy, precision, recall, f1

Execution Steps

In [ ]:
# Step 1: Download Dataset
CONFIG["data_dir"] = download_data(CONFIG)

Using Colab cache for faster access to the 'cifake-real-and-ai-generated-synthetic-images' dataset.
Dataset ready!


In [ ]:
# Step 2: Get DataLoaders with Enhanced Augmentation
train_loader, test_loader, class_to_idx = get_dataloaders(CONFIG)


 Dataset Statistics:
  Training samples: 100,000
  Testing samples: 20,000
  Classes: ['FAKE', 'REAL']


In [ ]:
# Step 3: Build Model with Enhanced LoRA
model = build_model(CONFIG, DEVICE)


  Building model...
  Loading google/vit-base-patch16-224...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Configuring LoRA...
  LoRA applied successfully

 Model ready for training!
  Device:           cuda
  Model:            google/vit-base-patch16-224
  Num classes:      2
  LoRA rank (r):    32
  LoRA alpha:       64
  ├─ Trainable params: 2,360,834 (2.678%)
  ├─ Frozen params:    85,800,194 (97.322%)
  └─ Total params:     88,161,028


In [ ]:
# Step 4: Train with All Optimizations
trained_model = train_model(
    model,
    train_loader,
    test_loader,
    CONFIG,
    DEVICE,
    class_to_idx,
    LABEL_MAP
)


 Starting Training
  Epochs: 5
  Batch size: 32
  Gradient accumulation: 2
  Effective batch size: 64
  Early stopping patience: 3


 Epoch 1/5:   0%|          | 0/3125 [00:00<?, ?it/s]

 Validating Epoch 1/5:   0%|          | 0/625 [00:00<?, ?it/s]


 Epoch [1/5] — 1165.2s
  Train → Loss: 0.2814 | Acc: 0.8643
  Val   → Loss: 0.1090 | Acc: 0.9635
  Val   → Precision: 0.9635 | Recall: 0.9635 | F1: 0.9634
   New best F1: 0.9634 - Model saved!


 Epoch 2/5:   0%|          | 0/3125 [00:00<?, ?it/s]

...


 Epoch [4/5] — 1164.4s
  Train → Loss: 0.0555 | Acc: 0.9796
  Val   → Loss: 0.0782 | Acc: 0.9690
  Val   → Precision: 0.9704 | Recall: 0.9690 | F1: 0.9690
   Patience: 2/3


 Epoch 5/5:   0%|          | 0/3125 [00:00<?, ?it/s]

 Validating Epoch 5/5:   0%|          | 0/625 [00:00<?, ?it/s]


 Epoch [5/5] — 1164.8s
  Train → Loss: 0.0451 | Acc: 0.9832
  Val   → Loss: 0.0416 | Acc: 0.9844
  Val   → Precision: 0.9845 | Recall: 0.9844 | F1: 0.9844


   New best F1: 0.9844 - Model saved!

 Training Complete!
  Best model saved to: ./saved_model_optimized/best_model
  Final model saved to: ./saved_model_optimized/final_model
  Best validation F1: 0.9844


In [ ]:
# Step 5: Load Best Model for Final Evaluation
from transformers import ViTForImageClassification
from peft import PeftModel

best_model_path = os.path.join(CONFIG["save_dir"], "best_model")
base_model = ViTForImageClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=CONFIG["num_classes"],
    ignore_mismatched_sizes=True
)
best_model = PeftModel.from_pretrained(base_model, best_model_path)
best_model = best_model.to(DEVICE)

print(f"Loaded best model from {best_model_path}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Loaded best model from ./saved_model_optimized/best_model


In [ ]:
# Step 6: Final Comprehensive Evaluation
final_accuracy, final_precision, final_recall, final_f1 = final_evaluation(
    best_model,
    test_loader,
    DEVICE,
    LABEL_MAP
)


 FINAL EVALUATION


Evaluating:   0%|          | 0/625 [00:00<?, ?it/s]


 Overall Metrics:
  Accuracy:  0.9844 (98.44%)
  Precision: 0.9845
  Recall:    0.9844
  F1 Score:  0.9844

 Detailed Classification Report:
              precision    recall  f1-score   support

        FAKE     0.9773    0.9918    0.9845     10000
        REAL     0.9917    0.9770    0.9843     10000

    accuracy                         0.9844     20000
   macro avg     0.9845    0.9844    0.9844     20000
weighted avg     0.9845    0.9844    0.9844     20000



In [ ]:
import os

# Define a path to save the model locally
local_save_path = "./my_local_model"

# Create the directory if it doesn't exist
os.makedirs(local_save_path, exist_ok=True)

# Save the best model
best_model.save_pretrained(local_save_path)

print(f"Model successfully saved to: {local_save_path}")
print("You can now download the 'my_local_model' folder from the Colab file browser.")

Model successfully saved to: ./my_local_model
You can now download the 'my_local_model' folder from the Colab file browser.


In [ ]:
# Zip the saved model directory
zip_filename = "my_local_model.zip"
shutil.make_archive(local_save_path, 'zip', local_save_path)
print(f"Model zipped to {zip_filename}")

Model zipped to my_local_model.zip


```markdown
Now, run the following cell to download the zipped model directly to your local machine.
```

In [ ]:
from google.colab import files

# Trigger browser download
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>